# MoonBoard Edge-Aware GAT Grade Prediction

This notebook is a submission-oriented, self-contained version of the edge-aware GAT experiment. It predicts MoonBoard 2016 grade classes from each problem represented as an individual directed hold graph.

Unlike the earlier ensemble experiment, this notebook trains and evaluates only the custom edge-aware GAT branch.


## Required Files

Place these files in the same folder as this notebook:

- `moonGen_scrape_2016_final`: MoonBoardRNN raw MoonBoard 2016 pickle dataset.

The model does not use `user_grade`. It does use route metadata available in the raw dataset, including repeats, benchmark/master flags, and setter id.


In [ ]:
# Optional: uncomment in Colab if dependencies are missing.
# !pip install -q torch scikit-learn scipy pandas tqdm


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
DATA_PATH = ROOT / "moonGen_scrape_2016_final"
print("working directory:", ROOT)
print("dataset exists:", DATA_PATH.exists(), DATA_PATH)


## Configuration

The default configuration runs the full dataset. For a quick smoke test, reduce `max_samples`, `gat_epochs`, and `gat_seeds`.


In [ ]:
from types import SimpleNamespace

CONFIG = SimpleNamespace(
    data=DATA_PATH,
    out=Path("runs_edge_aware_gat_only"),
    seed=471,
    max_samples=0,               # 0 means full dataset
    test_size=0.20,
    val_size_in_trainval=0.20,
    pair_hash_dim=32768,
    setter_hash_dim=4096,
    gat_epochs=100,
    gat_hidden=160,
    gat_seeds=3,
    gat_batch_size=128,
    gat_lr=8e-4,
    gat_weight_decay=1e-4,
    gat_dropout=0.25,
    ordinal_weight=0.02,
    gat_log_every=10,
    device="auto",
)


## Core Implementation

This cell defines the dataset loader, graph/relation feature extraction, edge-aware GAT layers, training loop, and evaluation metrics. Relation statistics are computed from the training split only.


In [ ]:
#!/usr/bin/env python3
"""
Edge-aware GAT-only MoonBoard grade prediction notebook core.

This cell defines data loading, train-only relation statistics, route-level dense features,
and the custom edge-aware GAT model. It intentionally excludes the classical ML ensemble
and meta-stacking pipeline used in exploratory experiments.
"""
from __future__ import annotations

import argparse
import json
import math
import pickle
import random
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False
    torch = None
    nn = None
    F = None
    Dataset = object
    DataLoader = None

from tqdm import tqdm

warnings.filterwarnings("ignore")

GRADE_MAP = {
    "6B": 0, "6B+": 0,
    "6C": 1, "6C+": 1,
    "7A": 2,
    "7A+": 3,
    "7B": 4, "7B+": 4,
    "7C": 5,
    "7C+": 6,
    "8A": 7,
    "8A+": 8,
    "8B": 9,
}
NUM_CLASSES = 10
BOARD_X = 11
BOARD_Y = 18
N_POS = BOARD_X * BOARD_Y


def log(msg: str) -> None:
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


@dataclass(frozen=True)
class Problem:
    key: str
    label: int
    grade: str
    start: tuple
    mid: tuple
    end: tuple
    repeats: float
    is_benchmark: bool
    is_master: bool
    setter_id: str

    @property
    def holds(self):
        return self.start + self.mid + self.end

    @property
    def typed(self):
        return tuple([(h, 's') for h in self.start] + [(h, 'm') for h in self.mid] + [(h, 'e') for h in self.end])


def load_problems(path: Path) -> List[Problem]:
    raw = pickle.load(Path(path).open('rb'))
    out: List[Problem] = []
    items = raw.items() if hasattr(raw, 'items') else enumerate(raw)
    for k, v in items:
        if not isinstance(v, dict):
            continue
        grade = v.get('grade')
        if grade not in GRADE_MAP:
            continue
        setter = v.get('setter') or {}
        if not isinstance(setter, dict):
            setter = {}
        out.append(Problem(
            key=str(k),
            label=int(GRADE_MAP[grade]),
            grade=str(grade),
            start=tuple(tuple(map(int, h)) for h in v.get('start', []) or []),
            mid=tuple(tuple(map(int, h)) for h in v.get('mid', []) or []),
            end=tuple(tuple(map(int, h)) for h in v.get('end', []) or []),
            repeats=float(v.get('repeats') or 0.0),
            is_benchmark=bool(v.get('is_benchmark')),
            is_master=bool(v.get('is_master')),
            setter_id=str(setter.get('Id') or setter.get('Nickname') or 'NONE'),
        ))
    return out


def hold_col(h) -> int:
    return int(h[0]) * BOARD_Y + int(h[1])


def role_id(t: str) -> int:
    return {'s': 0, 'm': 1, 'e': 2}[t]


def phash_int(vals, mod: int) -> int:
    x = 2166136261
    for v in vals:
        x ^= int(v) & 0xffffffff
        x = (x * 16777619) & 0xffffffff
    return x % mod


def pair_keys(ha, ta, hb, tb):
    dx = hb[0] - ha[0]
    dy = hb[1] - ha[1]
    dist = math.hypot(dx, dy)
    return [
        ('dir', ha, hb),
        ('undir', tuple(sorted([ha, hb]))),
        ('typed', ta, tb, ha, hb),
        ('geom', ta, tb, min(6, abs(dx)), max(0, min(16, dy + 8)), min(10, int(round(dist)))),
    ]


def build_tables(problems: List[Problem], train_idx, smoothing: float = 12.0):
    log('build train-only relation/setter encodings')
    y = np.array([problems[int(i)].label for i in train_idx], dtype=np.int64)
    global_dist = np.bincount(y, minlength=NUM_CLASSES).astype(np.float64)
    global_dist /= max(1.0, global_dist.sum())
    global_mean = float(y.mean()) if len(y) else 0.0

    setter_count = defaultdict(int)
    setter_sum = defaultdict(float)
    setter_dist = defaultdict(lambda: np.zeros(NUM_CLASSES, dtype=np.float64))
    pair_count = defaultdict(int)
    pair_sum = defaultdict(float)
    pair_dist = defaultdict(lambda: np.zeros(NUM_CLASSES, dtype=np.float64))

    it = tqdm(train_idx, desc='tables', unit='problem')
    for ii in it:
        p = problems[int(ii)]
        setter_count[p.setter_id] += 1
        setter_sum[p.setter_id] += p.label
        setter_dist[p.setter_id][p.label] += 1
        typed = p.typed
        for i, (ha, ta) in enumerate(typed):
            for j, (hb, tb) in enumerate(typed):
                if i == j:
                    continue
                dx = hb[0] - ha[0]
                dy = hb[1] - ha[1]
                dist = math.hypot(dx, dy)
                if dist == 0 or dist > 10:
                    continue
                for key in pair_keys(ha, ta, hb, tb):
                    pair_count[key] += 1
                    pair_sum[key] += p.label
                    pair_dist[key][p.label] += 1

    def mean(sumtab, cnttab, key):
        c = cnttab.get(key, 0)
        return (sumtab.get(key, 0.0) + smoothing * global_mean) / (c + smoothing)

    def dist(tab, cnttab, key):
        c = cnttab.get(key, 0)
        return (tab.get(key, np.zeros(NUM_CLASSES, dtype=np.float64)) + smoothing * global_dist) / (c + smoothing)

    return dict(
        gm=global_mean,
        gd=global_dist,
        smoothing=smoothing,
        sc=dict(setter_count),
        ss=dict(setter_sum),
        sd=dict(setter_dist),
        pc=dict(pair_count),
        ps=dict(pair_sum),
        pdist=dict(pair_dist),
        mean=mean,
        dist=dist,
    )


def safe_mean(vals, default=0.0):
    return float(np.mean(vals)) if len(vals) else float(default)


def summarize(vals, default: float):
    if not vals:
        vals = [default]
    a = np.asarray(vals, dtype=np.float32)
    top = np.sort(a)[::-1]
    return [
        float(a.mean()), float(a.std()), float(a.min()), float(a.max()),
        float(np.quantile(a, 0.10)), float(np.quantile(a, 0.25)), float(np.quantile(a, 0.50)),
        float(np.quantile(a, 0.75)), float(np.quantile(a, 0.90)), float(np.quantile(a, 0.95)),
        float(top[:2].mean()), float(top[:3].mean()), float(top[:5].mean() if len(top) >= 5 else top.mean()),
    ]


def build_no_usergrade_features(problems: List[Problem], tables, pair_hash_dim=32768, setter_hash_dim=4096):
    """Build graph/relation features. user_grade is intentionally not read anywhere here."""
    log('build no-user-grade sparse+dense graph/relation features')
    rows, cols, vals = [], [], []
    dense = []

    typed_off = 0
    roleless_off = typed_off + 3 * N_POS
    pair_off = roleless_off + N_POS
    geom_pair_off = pair_off + pair_hash_dim
    setter_off = geom_pair_off + pair_hash_dim
    dim = setter_off + setter_hash_dim

    iterator = tqdm(range(len(problems)), desc='features(no_user_grade)', unit='problem')

    for r in iterator:
        p = problems[r]

        def add(c, v=1.0):
            rows.append(r)
            cols.append(int(c))
            vals.append(float(v))

        for h in p.start:
            add(typed_off + hold_col(h)); add(roleless_off + hold_col(h))
        for h in p.mid:
            add(typed_off + N_POS + hold_col(h)); add(roleless_off + hold_col(h))
        for h in p.end:
            add(typed_off + 2 * N_POS + hold_col(h)); add(roleless_off + hold_col(h))

        sid_hash = phash_int([ord(ch) for ch in str(p.setter_id)[:32]], setter_hash_dim)
        add(setter_off + sid_hash, 1.0)

        typed = p.typed
        rel_vals, geom_vals, conf_vals, edge_dy, edge_dx, dist_vals, adj_vals = [], [], [], [], [], [], []
        for i, (ha, ta) in enumerate(typed):
            for j, (hb, tb) in enumerate(typed):
                if i == j:
                    continue
                dx = hb[0] - ha[0]
                dy = hb[1] - ha[1]
                dist = math.hypot(dx, dy)
                if dist == 0 or dist > 10:
                    continue
                salt = 19*role_id(ta) + 37*role_id(tb) + 7*min(6, abs(dx)) + 13*max(0, min(16, dy+8)) + 23*min(10, int(round(dist)))
                add(pair_off + phash_int([ha[0], ha[1], hb[0], hb[1], salt], pair_hash_dim), 1.0)
                uh = sorted([ha, hb])
                add(geom_pair_off + phash_int([uh[0][0], uh[0][1], uh[1][0], uh[1][1], salt], pair_hash_dim), 1.0)

                keys = pair_keys(ha, ta, hb, tb)
                encs = [tables['mean'](tables['ps'], tables['pc'], k) for k in keys]
                cnts = [tables['pc'].get(k, 0) for k in keys]
                enc = float(np.mean(encs))
                conf = float(np.log1p(max(cnts)))
                geom = 0.13 * abs(dx) + 0.11 * max(0, dy) + 0.04 * dist
                rel_vals.append(enc + geom)
                geom_vals.append(geom)
                conf_vals.append(conf)
                edge_dx.append(dx); edge_dy.append(dy); dist_vals.append(dist)
                if abs(i - j) == 1:
                    adj_vals.append(enc + geom)

        holds = p.holds
        xs = [h[0] for h in holds] or [0]
        ys = [h[1] for h in holds] or [0]
        c = tables['sc'].get(p.setter_id, 0)
        setter_mean = (tables['ss'].get(p.setter_id, 0.0) + tables['smoothing'] * tables['gm']) / (c + tables['smoothing'])
        setter_dist = tables['dist'](tables['sd'], tables['sc'], p.setter_id)

        dense.append([
            math.log1p(float(p.repeats)), float(p.is_benchmark), float(p.is_master),
            float(c), float(setter_mean), *setter_dist.tolist(),
            len(p.start), len(p.mid), len(p.end), len(holds),
            np.mean(xs), np.std(xs), min(xs), max(xs), np.mean(ys), np.std(ys), min(ys), max(ys),
            max(xs) - min(xs), max(ys) - min(ys),
            *summarize(rel_vals, tables['gm']),
            *summarize(geom_vals, 0.0),
            *summarize(conf_vals, 0.0),
            *summarize(adj_vals, tables['gm']),
            safe_mean([abs(x) for x in edge_dx]),
            safe_mean([max(0, y) for y in edge_dy]),
            safe_mean(dist_vals),
            float(np.mean([d > 4 for d in dist_vals])) if dist_vals else 0.0,
            float(np.mean([y > 3 for y in edge_dy])) if edge_dy else 0.0,
        ])

    X = sparse.csr_matrix((vals, (rows, cols)), shape=(len(problems), dim), dtype=np.float32)
    D = np.asarray(dense, dtype=np.float32)
    D = np.nan_to_num(D, nan=0.0, posinf=0.0, neginf=0.0)
    return X, D


def sanitize_probs(P: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    P = np.asarray(P, dtype=np.float32)
    P = np.nan_to_num(P, nan=0.0, posinf=0.0, neginf=0.0)
    if P.ndim != 2 or P.shape[1] != NUM_CLASSES:
        raise ValueError(f"expected [n,{NUM_CLASSES}], got {P.shape}")
    P[P < 0] = 0.0
    s = P.sum(axis=1, keepdims=True)
    bad = (~np.isfinite(s[:, 0])) | (s[:, 0] <= eps)
    if np.any(bad):
        P[bad] = 1.0 / NUM_CLASSES
        s = P.sum(axis=1, keepdims=True)
    return P / np.clip(s, eps, None)


def probs_align(clf, X) -> np.ndarray:
    raw = clf.predict_proba(X)
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    out = np.zeros((raw.shape[0], NUM_CLASSES), dtype=np.float32)
    for j, c in enumerate(clf.classes_):
        ci = int(c)
        if 0 <= ci < NUM_CLASSES:
            out[:, ci] = raw[:, j]
    return sanitize_probs(out)


def metric(y, pred, prob=None):
    y = np.asarray(y)
    pred = np.asarray(pred)
    out = {
        'exact_acc': float(accuracy_score(y, pred)),
        'micro_f1': float(f1_score(y, pred, average='micro', zero_division=0)),
        'relaxed_acc': float(np.mean(np.abs(y - pred) <= 1)),
        'within2_acc': float(np.mean(np.abs(y - pred) <= 2)),
        'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y, pred, average='weighted', zero_division=0)),
        'mse_argmax': float(np.mean((y - pred) ** 2)),
        'mae_argmax': float(np.mean(np.abs(y - pred))),
    }
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    rare = [5, 6, 7, 8, 9]
    out['rare_f1_5_9'] = float(np.mean([f1[i] for i in rare]))
    out['rare_recall_5_9'] = float(np.mean([rc[i] for i in rare]))
    out['rare_precision_5_9'] = float(np.mean([pr[i] for i in rare]))
    if prob is not None:
        exp = sanitize_probs(prob) @ np.arange(NUM_CLASSES)
        out['mse_expected'] = float(np.mean((exp - y) ** 2))
        out['mae_expected'] = float(np.mean(np.abs(exp - y)))
    return out


def graph_knn_probs(Xtr, ytr, Xev, k, batch=2048):
    nnm = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute', n_jobs=-1).fit(Xtr)
    out = np.zeros((Xev.shape[0], NUM_CLASSES), dtype=np.float32)
    rng = range(0, Xev.shape[0], batch)
    rng = tqdm(rng, total=len(range(0, Xev.shape[0], batch)), desc=f'knn{k}', unit='batch')
    for s in rng:
        e = min(s + batch, Xev.shape[0])
        dist, ind = nnm.kneighbors(Xev[s:e], return_distance=True)
        w = np.clip(1.0 - dist, 0, 1) + 1e-4
        for r in range(e - s):
            labs = ytr[ind[r]]
            for jj, lab in enumerate(labs):
                out[s + r, int(lab)] += float(w[r, jj])
            out[s + r] /= max(1e-9, out[s + r].sum())
    return sanitize_probs(out)


def exact_bias_search(Pv, yv, Pt, trials=30000, seed=471, scale=0.28):
    rng = np.random.default_rng(seed)
    lv = np.log(np.clip(sanitize_probs(Pv), 1e-9, 1.0))
    lt = np.log(np.clip(sanitize_probs(Pt), 1e-9, 1.0))
    best_score = -1.0
    best_bias = np.zeros(NUM_CLASSES, dtype=np.float32)
    it = tqdm(range(trials), desc='bias', unit='trial')
    for _ in it:
        b = rng.normal(0.0, scale, size=NUM_CLASSES).astype(np.float32)
        score = accuracy_score(yv, (lv + b).argmax(axis=1))
        if score > best_score:
            best_score = float(score)
            best_bias = b
    Pt2 = np.exp(lt + best_bias)
    return best_score, sanitize_probs(Pt2)


def random_blend(bank: Dict[str, Dict[str, np.ndarray]], yv: np.ndarray, trials: int, seed: int):
    names = list(bank.keys())
    if len(names) < 2 or trials <= 0:
        return None
    rng = np.random.default_rng(seed)
    best = None
    it = tqdm(range(trials), desc='blend', unit='trial')
    for _ in it:
        m = int(rng.integers(2, min(9, len(names)) + 1))
        chosen = list(rng.choice(names, size=m, replace=False))
        w = rng.dirichlet(np.ones(m) * 0.45)
        Pv = sum(float(a) * bank[n]['val'] for a, n in zip(w, chosen))
        score = accuracy_score(yv, Pv.argmax(axis=1))
        if best is None or score > best['val_exact']:
            Pt = sum(float(a) * bank[n]['test'] for a, n in zip(w, chosen))
            best = {'val_exact': float(score), 'members': chosen, 'weights': w.tolist(), 'test': sanitize_probs(Pt)}
    return best


# ----------------------------- custom edge-aware GAT -----------------------------

if TORCH_AVAILABLE:
    @dataclass
    class GraphItem:
        x: torch.Tensor
        edge_attr: torch.Tensor
        edge_mask: torch.Tensor
        dense: np.ndarray
        y: int
        idx: int

    class GraphDataset(Dataset):
        def __init__(self, graphs, indices):
            self.graphs = graphs
            self.indices = list(map(int, indices))
        def __len__(self):
            return len(self.indices)
        def __getitem__(self, i):
            return self.graphs[self.indices[i]]

    def collate_graphs(items):
        B = len(items)
        N = max(it.x.shape[0] for it in items)
        Fd = items[0].x.shape[1]
        Ed = items[0].edge_attr.shape[-1]
        x = torch.zeros(B, N, Fd)
        ea = torch.zeros(B, N, N, Ed)
        em = torch.zeros(B, N, N, dtype=torch.bool)
        nm = torch.zeros(B, N, dtype=torch.bool)
        dense = torch.tensor(np.stack([it.dense for it in items]), dtype=torch.float32)
        y = torch.tensor([it.y for it in items], dtype=torch.long)
        for b, it in enumerate(items):
            n = it.x.shape[0]
            x[b, :n] = it.x
            ea[b, :n, :n] = it.edge_attr
            em[b, :n, :n] = it.edge_mask
            nm[b, :n] = True
        return {'x': x, 'edge_attr': ea, 'edge_mask': em, 'node_mask': nm, 'dense': dense, 'y': y}

    def build_graphs(problems, tables, dense_scaled, max_dist=10.0):
        log('build custom no-user-grade edge-aware GAT graphs')
        graphs = []
        edge_dim = 25
        it = tqdm(enumerate(problems), total=len(problems), desc='gat_graphs_no_user_grade', unit='problem')
        for idx, p in it:
            typed = list(p.typed)
            xs = []
            for h, t in typed:
                typ = role_id(t)
                xs.append([
                    h[0] / 10.0, h[1] / 17.0,
                    float(typ == 0), float(typ == 1), float(typ == 2),
                    len(p.start) / 4.0, len(p.mid) / 12.0, len(p.end) / 4.0,
                ])
            if not xs:
                xs = [[0.0] * 8]
                typed = [((0, 0), 'm')]
            n = len(xs)
            edge_attr = np.zeros((n, n, edge_dim), dtype=np.float32)
            edge_mask = np.zeros((n, n), dtype=bool)
            for i, (ha, ta) in enumerate(typed):
                for j, (hb, tb) in enumerate(typed):
                    dx = hb[0] - ha[0]
                    dy = hb[1] - ha[1]
                    distv = math.hypot(dx, dy)
                    if i == j:
                        attr = [0.0] * edge_dim
                        attr[7] = 1.0
                        edge_attr[i, j] = attr
                        edge_mask[i, j] = True
                        continue
                    if distv <= 0 or distv > max_dist:
                        continue
                    keys = pair_keys(ha, ta, hb, tb)
                    means = [tables['mean'](tables['ps'], tables['pc'], k) for k in keys]
                    dists = [tables['dist'](tables['pdist'], tables['pc'], k) for k in keys]
                    enc = float(np.mean(means)) / 9.0
                    conf = float(np.mean([min(1.0, tables['pc'].get(k, 0) / 20.0) for k in keys]))
                    geom_hard = min(1.0, (0.13 * abs(dx) + 0.11 * max(0, dy) + 0.04 * distv) / 2.2)
                    st = role_id(ta)
                    dt = role_id(tb)
                    dist_mean = np.mean(dists, axis=0)
                    attr = [
                        dx / 10.0, dy / 17.0, abs(dx) / 10.0, abs(dy) / 17.0, distv / max_dist,
                        float(dy > 0), float(dy < 0), float(abs(dx) >= 4),
                        float(st == 0), float(st == 1), float(st == 2), float(dt == 0),
                        enc, conf, geom_hard,
                        *dist_mean.tolist(),
                    ]
                    edge_attr[i, j] = np.asarray(attr, dtype=np.float32)
                    edge_mask[i, j] = True
            graphs.append(GraphItem(
                torch.tensor(xs, dtype=torch.float32),
                torch.tensor(edge_attr, dtype=torch.float32),
                torch.tensor(edge_mask, dtype=torch.bool),
                dense_scaled[idx].astype(np.float32),
                int(p.label),
                idx,
            ))
        return graphs, graphs[0].x.shape[1], graphs[0].edge_attr.shape[-1]

    class EdgeAwareGATLayer(nn.Module):
        def __init__(self, hidden, edge_dim, dropout):
            super().__init__()
            self.lin = nn.Linear(hidden, hidden, bias=False)
            self.edge = nn.Linear(edge_dim, hidden, bias=False)
            self.a_src = nn.Parameter(torch.empty(hidden))
            self.a_dst = nn.Parameter(torch.empty(hidden))
            self.a_edge = nn.Parameter(torch.empty(hidden))
            self.out = nn.Linear(hidden, hidden)
            self.norm = nn.LayerNorm(hidden)
            self.drop = nn.Dropout(dropout)
            self.reset_parameters()
        def reset_parameters(self):
            self.lin.reset_parameters(); self.edge.reset_parameters(); self.out.reset_parameters()
            nn.init.xavier_uniform_(self.a_src.view(1, -1))
            nn.init.xavier_uniform_(self.a_dst.view(1, -1))
            nn.init.xavier_uniform_(self.a_edge.view(1, -1))
        def forward(self, h, ea, em, nm):
            z = self.lin(h)
            e = self.edge(ea)
            score = (z.unsqueeze(2) * self.a_src).sum(-1) + (z.unsqueeze(1) * self.a_dst).sum(-1) + (e * self.a_edge).sum(-1)
            score = F.leaky_relu(score, 0.2).masked_fill(~em, -1e9)
            alpha = torch.softmax(score, dim=1).masked_fill(~em, 0.0)
            msg = z.unsqueeze(2) + e
            out = (alpha.unsqueeze(-1) * msg).sum(dim=1)
            out = self.out(self.drop(out))
            return self.norm(h + out) * nm.unsqueeze(-1).float()

    class CustomGAT(nn.Module):
        def __init__(self, node_dim, edge_dim, dense_dim, hidden, dropout):
            super().__init__()
            self.node = nn.Linear(node_dim, hidden)
            self.g1 = EdgeAwareGATLayer(hidden, edge_dim, dropout)
            self.g2 = EdgeAwareGATLayer(hidden, edge_dim, dropout)
            self.graph_head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(dropout))
            self.dense_head = nn.Sequential(nn.Linear(dense_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, hidden), nn.ReLU())
            self.cls = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, NUM_CLASSES))
        def forward(self, b, device):
            x = b['x'].to(device)
            ea = b['edge_attr'].to(device)
            em = b['edge_mask'].to(device)
            nm = b['node_mask'].to(device)
            d = b['dense'].to(device)
            h = F.elu(self.node(x)) * nm.unsqueeze(-1).float()
            h = F.elu(self.g1(h, ea, em, nm))
            h = F.elu(self.g2(h, ea, em, nm))
            denom = nm.sum(dim=1, keepdim=True).clamp_min(1).float()
            mean = h.sum(dim=1) / denom
            mx = h.masked_fill(~nm.unsqueeze(-1), -1e9).max(dim=1).values
            mx = torch.where(torch.isfinite(mx), mx, torch.zeros_like(mx))
            gh = self.graph_head(torch.cat([mean, mx], dim=1))
            dh = self.dense_head(d)
            return self.cls(torch.cat([gh, dh], dim=1))

    def eval_gat_model(model, loader, device):
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for b in loader:
                logits = model(b, device)
                ps.append(F.softmax(logits, dim=1).cpu().numpy())
                ys.extend(b['y'].numpy().tolist())
        return np.array(ys, dtype=np.int64), sanitize_probs(np.vstack(ps))

    def train_gat(name, graphs, tr, va, te, node_dim, edge_dim, dense_dim, args, seed):
        log(f'train {name}')
        seed_all(seed)
        device = torch.device(args.device if args.device != 'auto' else ('cuda' if torch.cuda.is_available() else 'cpu'))
        model = CustomGAT(node_dim, edge_dim, dense_dim, args.gat_hidden, args.gat_dropout).to(device)
        ytr = np.array([graphs[int(i)].y for i in tr])
        counts = np.bincount(ytr, minlength=NUM_CLASSES).astype(np.float32)
        w = np.ones(NUM_CLASSES, dtype=np.float32)
        present = counts > 0
        w[present] = np.sqrt(len(ytr) / (present.sum() * counts[present]))
        w[present] /= max(1e-9, w[present].mean())
        ce = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
        opt = torch.optim.AdamW(model.parameters(), lr=args.gat_lr, weight_decay=args.gat_weight_decay)
        tr_loader = DataLoader(GraphDataset(graphs, tr), batch_size=args.gat_batch_size, shuffle=True, collate_fn=collate_graphs)
        va_loader = DataLoader(GraphDataset(graphs, va), batch_size=args.gat_batch_size, shuffle=False, collate_fn=collate_graphs)
        te_loader = DataLoader(GraphDataset(graphs, te), batch_size=args.gat_batch_size, shuffle=False, collate_fn=collate_graphs)
        best = None
        with tqdm(range(1, args.gat_epochs + 1), unit='epoch', desc=name) as pbar:
            pbar.clear()
            for ep in pbar:
                model.train()
                total = 0.0
                nobs = 0
                for b in tr_loader:
                    opt.zero_grad(set_to_none=True)
                    logits = model(b, device)
                    yy = b['y'].to(device)
                    prob = F.softmax(logits, dim=1)
                    exp = (prob * torch.arange(NUM_CLASSES, device=device).float()).sum(1)
                    loss = ce(logits, yy) + args.ordinal_weight * F.mse_loss(exp, yy.float())
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    opt.step()
                    total += float(loss.item()) * len(yy)
                    nobs += len(yy)
                yv, pv = eval_gat_model(model, va_loader, device)
                vexact = accuracy_score(yv, pv.argmax(1))
                if best is None or vexact > best[0]:
                    yt, pt = eval_gat_model(model, te_loader, device)
                    best = (vexact, ep, pv.copy(), pt.copy())
                current_loss = total / max(1, nobs)
                postfix_new = ", ".join([
                    f"loss: {current_loss:.4f}",
                    f"val_exact: {vexact:.4f} (best: {best[0]:.4f})",
                    f"best_epoch: {best[1]}",
                ])
                pbar.set_postfix_str(postfix_new)
        print(f"Best Epoch ({name}): {best[1]}")
        print(f"Best Validation Exact Accuracy ({name}): {best[0]:.4f}")
        return {'name': name, 'val_exact': best[0], 'epoch': best[1], 'val_probs': best[2], 'test_probs': best[3]}


## Edge-Aware GAT Pipeline

This runner keeps only the edge-aware GAT branch:

1. Load MoonBoard problems.
2. Create train/validation/test split.
3. Build train-only pair/setter relation statistics.
4. Build route-level dense features and edge-attributed hold graphs.
5. Train one or more edge-aware GAT seeds.
6. Select models by validation exact accuracy and report test metrics.


In [ ]:
def run_edge_aware_gat_only(args):
    seed_all(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)

    log("EDGE-AWARE GAT ONLY: no classical ML stack, no meta ensemble")
    log("NO USER_GRADE: disabled as feature, override, routing, and diagnostics")
    log(f"load {args.data}")
    problems = load_problems(args.data)
    if args.max_samples and args.max_samples < len(problems):
        y0 = np.array([p.label for p in problems])
        try:
            _, sub = train_test_split(np.arange(len(problems)), test_size=args.max_samples, random_state=args.seed, stratify=y0)
        except ValueError:
            rng = np.random.default_rng(args.seed)
            sub = rng.choice(len(problems), size=args.max_samples, replace=False)
        problems = [problems[int(i)] for i in sub]
        log(f"max_samples={len(problems)}")

    y = np.array([p.label for p in problems], dtype=np.int64)
    all_idx = np.arange(len(problems))
    trainval, test = train_test_split(all_idx, test_size=args.test_size, random_state=args.seed, stratify=y)
    train, val = train_test_split(trainval, test_size=args.val_size_in_trainval, random_state=args.seed + 1, stratify=y[trainval])
    log(f"split train={len(train)} val={len(val)} test={len(test)} labels={dict(Counter(y.tolist()))}")

    tables = build_tables(problems, train)
    _, dense = build_no_usergrade_features(problems, tables, args.pair_hash_dim, args.setter_hash_dim)
    dense = np.nan_to_num(dense, nan=0.0, posinf=0.0, neginf=0.0)
    scaler = StandardScaler().fit(dense[train])
    dense_scaled = scaler.transform(dense).astype(np.float32)
    log(f"dense route features: {dense_scaled.shape}")

    if not TORCH_AVAILABLE:
        raise RuntimeError("torch is required for the edge-aware GAT notebook")

    graphs, node_dim, edge_dim = build_graphs(problems, tables, dense_scaled)
    rows = []
    best = None

    for i in range(args.gat_seeds):
        seed = args.seed + 101 * i
        result = train_gat(f"edge_gat_seed{seed}", graphs, train, val, test, node_dim, edge_dim, dense_scaled.shape[1], args, seed)
        y_test = np.array([graphs[int(j)].y for j in test], dtype=np.int64)
        row = {
            "model": result["name"],
            "selected_by": "val_exact",
            "val_exact": float(result["val_exact"]),
            "epoch": int(result["epoch"]),
            **metric(y_test, result["test_probs"].argmax(axis=1), result["test_probs"]),
        }
        rows.append(row)
        if best is None or row["val_exact"] > best["val_exact"]:
            best = row

    df = pd.DataFrame(rows).sort_values(["val_exact", "exact_acc", "relaxed_acc"], ascending=False)
    leaderboard_path = args.out / "edge_aware_gat_only_leaderboard.csv"
    best_path = args.out / "edge_aware_gat_only_best_summary.json"
    config_path = args.out / "config.json"
    df.to_csv(leaderboard_path, index=False)
    best = df.iloc[0].to_dict()
    best_path.write_text(json.dumps(best, indent=2, ensure_ascii=False), encoding="utf-8")
    config_path.write_text(json.dumps(vars(args), indent=2, default=str, ensure_ascii=False), encoding="utf-8")

    log("edge-aware GAT leaderboard:")
    with pd.option_context("display.max_columns", None, "display.width", 220):
        print(df.to_string(index=False), flush=True)
    log(f"saved {leaderboard_path}")
    return df, best


## Run Training

Run the next cell to train and evaluate the edge-aware GAT model.


In [ ]:
results_df, best_result = run_edge_aware_gat_only(CONFIG)
display(results_df)
print("Best validation-selected result:")
print(json.dumps(best_result, indent=2, ensure_ascii=False))


## Plot Results

This cell gives a compact visual summary of the seed-level runs.


In [ ]:
import matplotlib.pyplot as plt

metrics = ["exact_acc", "relaxed_acc", "macro_f1", "rare_f1_5_9"]
fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 3.5), sharey=False)
for ax, metric_name in zip(axes, metrics):
    ax.bar(results_df["model"], results_df[metric_name])
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
